<a href="https://colab.research.google.com/github/abdulmusai/Breast-Cancer-Diagnostic-Profiles-using-K-Nearest-Neighbor-Classification/blob/main/Breast_Cancer_Diagnostic_Profiles_using_KNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# Pattern Recognition Mini Project: Supervised K-Nearest Neighbor Classification
# Corrected Pipeline: Transition from Unsupervised Clustering to Supervised KNN
# File: knn_classification.py
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Configure visualization environment for publication-ready outputs
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
np.random.seed(42)

print("=== 1. Supervised Learning Data Ingestion & Preprocessing ===")
file_path = "/content/drive/MyDrive/breast-cancer-body-clean-1.csv"

try:
    # Skip the corrupted header metadata tracker and read using 'latin1' encoding
    df_raw = pd.read_csv(file_path, skiprows=1, header=None, encoding='latin1')

    # Assign explicit columns as required by the rubric tracking matrix
    clean_columns = [
        "id", "diagnosis", "radius_mean", "texture_mean", "perimeter_mean",
        "area_mean", "smoothness_mean", "compactness_mean", "concavity_mean",
        "concavity_worst", "concave_points_worst", "symmetry_worst", "fractal_dimension_worst"
    ]
    df_raw.columns = clean_columns

    # Drop the unused, empty 'id' row tracking field
    df = df_raw.copy()
    print(f"Dataset successfully loaded. Records available: {df.shape[0]}")
except Exception as e:
    print(f"Ingestion Error: Failed to open or parse the file correctly. Details: {e}")
    exit()

# Pivot to Supervised Learning: Map target column explicitly to numeric bits
df['target'] = df['diagnosis'].map({'M': 1, 'B': 0})

# Isolate feature predictors matrix (X) and label vector (y)
X = df.drop(columns=['id', 'diagnosis', 'target'])
y = df['target']

print("\n--- Target Class Balance Summary ---")
print(df['diagnosis'].value_counts())
print(df['diagnosis'].value_counts(normalize=True))


=== 1. Supervised Learning Data Ingestion & Preprocessing ===
Dataset successfully loaded. Records available: 100

--- Target Class Balance Summary ---
diagnosis
M    65
B    35
Name: count, dtype: int64
diagnosis
M    0.65
B    0.35
Name: proportion, dtype: float64


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ------------------------------------------------------------------------------
# Task 3 & 4: Stratified Train/Test Split & Feature Standardization
# ------------------------------------------------------------------------------
print("\n=== 2. Stratified Partitioning & Z-Score Scaling ===")
# Establish an 80/20 train/test split, stratified to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training Subset Dimensions: {X_train.shape[0]} rows | Evaluation Test Dimensions: {X_test.shape[0]} rows")

# Apply StandardScaler to guarantee unbiased distance evaluations across dimensions
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



=== 2. Stratified Partitioning & Z-Score Scaling ===
Training Subset Dimensions: 80 rows | Evaluation Test Dimensions: 20 rows


In [4]:
# ------------------------------------------------------------------------------
# Task 5, 6 & 7: KNN Hyperparameter Tuning via Cross-Validation
# ------------------------------------------------------------------------------
print("\n=== 3. Neighbors Hyperparameter Optimization Tuning ===")
k_values = [1, 3, 5, 7, 9, 11]
cv_scores = []

# Use 5-fold cross-validation to isolate accuracy profile behavior
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())
    print(f"Neighbors Evaluated: k = {k:02d} | Mean Validation Accuracy: {scores.mean():.4f}")

# Extract peak performing parameter
optimal_k = k_values[np.argmax(cv_scores)]
print(f"\n[DECISION]: Peak verification performance achieved at optimal k = {optimal_k}")

# Plot hyperparameter optimization curve
plt.figure(figsize=(8, 5))
plt.plot(k_values, cv_scores, marker='s', linestyle='-', color='teal', linewidth=2, markersize=8)
plt.axvline(x=optimal_k, color='crimson', linestyle=':', label=f'Optimal Selection (k={optimal_k})')
plt.xlabel('Number of Neighbors (k)')
plt.ylabel('5-Fold Validation Accuracy')
plt.title('KNN Classification Hyperparameter Search Grid')
plt.legend()
plt.tight_layout()
plt.savefig('knn_tuning_curve.png', bbox_inches='tight')
plt.close()



=== 3. Neighbors Hyperparameter Optimization Tuning ===
Neighbors Evaluated: k = 01 | Mean Validation Accuracy: 0.9250
Neighbors Evaluated: k = 03 | Mean Validation Accuracy: 0.9375
Neighbors Evaluated: k = 05 | Mean Validation Accuracy: 0.9625
Neighbors Evaluated: k = 07 | Mean Validation Accuracy: 0.9500
Neighbors Evaluated: k = 09 | Mean Validation Accuracy: 0.9500
Neighbors Evaluated: k = 11 | Mean Validation Accuracy: 0.9500

[DECISION]: Peak verification performance achieved at optimal k = 5


In [5]:
# ------------------------------------------------------------------------------
# Task 8: Final Model Evaluation & Classification Analytics
# ------------------------------------------------------------------------------
print("\n=== 4. Final Classification Performance Assessment ===")
final_model = KNeighborsClassifier(n_neighbors=optimal_k)
final_model.fit(X_train_scaled, y_train)

# Generate predictions over holdout instances
y_pred = final_model.predict(X_test_scaled)

print(f"Holdout Generalization Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("--- Supervised Classification Performance Report ---")
print(classification_report(y_test, y_pred, target_names=['Benign (0)', 'Malignant (1)']))

# Construct confusion matrix visual output
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Benign', 'Malignant'], yticklabels=['Benign', 'Malignant'])
plt.ylabel('True Class')
plt.xlabel('Predicted Class')
plt.title(f'Confusion Matrix Map (k={optimal_k})')
plt.tight_layout()
plt.savefig('knn_confusion_matrix.png', bbox_inches='tight')
plt.close()



=== 4. Final Classification Performance Assessment ===
Holdout Generalization Accuracy: 0.8000

--- Supervised Classification Performance Report ---
               precision    recall  f1-score   support

   Benign (0)       0.80      0.57      0.67         7
Malignant (1)       0.80      0.92      0.86        13

     accuracy                           0.80        20
    macro avg       0.80      0.75      0.76        20
 weighted avg       0.80      0.80      0.79        20



In [6]:
# ------------------------------------------------------------------------------
# Task 9: Evaluation Inference Test on New Data Point
# ------------------------------------------------------------------------------
print("\n=== 5. Production Inference Testing over Unseen Samples ===")
# Simulating a new profile by extracting feature means
synthetic_case = np.array([X_train.mean(axis=0)])
scaled_case = scaler.transform(synthetic_case)

prediction = final_model.predict(scaled_case)[0]
probabilities = final_model.predict_proba(scaled_case)[0]

print(f"New Observation Diagnostics Result: {'MALIGNANT' if prediction == 1 else 'BENIGN'}")
print(f"Class Membership Probability Matrix: [Benign: {probabilities[0]:.2f}, Malignant: {probabilities[1]:.2f}]")



=== 5. Production Inference Testing over Unseen Samples ===
New Observation Diagnostics Result: MALIGNANT
Class Membership Probability Matrix: [Benign: 0.00, Malignant: 1.00]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
